# 軽量オープン LLM によるプロンプト演習とハルシネーション抑制

書籍 第1章 練習問題 4・5（自然言語による指示・ハルシネーション）および 第4章 練習問題 3（事前学習済み大規模言語モデル）に対応する演習ノートブックです。

API キーやアクセス申請（gated token）が不要で、Google Colab の無料枠 GPU（T4, 15GB VRAM）に収まるオープンモデル **`Qwen/Qwen2.5-1.5B-Instruct`** を使い、以下を体験します。

1. **プロンプトによる多様なタスク実行（感情分類・要約・英日翻訳）**
2. **ゼロショット（0-shot）vs フューショット（Few-shot）の定量比較**（`phrase-annotation.csv` のサブセットで出力形式の遵守率・Accuracy・Macro-F1 を測定）
3. **ハルシネーションの発生と根拠文書付与（RAG 風プロンプト）による抑制**（回答不能質問に対する拒否検証を含む）
4. **まとめ（深層学習・LLM の利点と課題）**

> **実行環境の目安**: Google Colab で **「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」** を選択して実行してください（モデルの読み込み〜全セル実行で約 2〜3 分）。

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU が利用可能です: {gpu_name} ({vram_gb:.1f} GB)")
else:
    print(
        "⚠️ GPU が検出されませんでした（CPU モードでは推論に時間がかかります）。\n"
        "Google Colab の場合は「ランタイム」→「ランタイムのタイプを変更」→「GPU（T4 など）」を選択して再実行してください。"
    )

In [ ]:
!pip install -q transformers accelerate pandas scikit-learn
!curl -fLO https://raw.githubusercontent.com/mamorlis/nlpbook/refs/heads/main/misc/phrase-annotation.csv

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()


def generate_response(messages, max_new_tokens=160, temperature=0.0):
    """チャット形式のメッセージ列から LLM の応答テキストを生成する。"""
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if temperature == 0.0:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


print(f"モデル読み込み完了: {MODEL_NAME} (device={model.device})")

---

## 1. プロンプトによる多様なタスク実行（分類・要約・翻訳）

従来の機械学習（第 2〜3 章）では、タスクごとに専用の教師データを用意して個別モデルを学習する必要がありました。
一方、指示チューニング（Instruction Tuning）された大規模言語モデル（LLM）では、**同じ 1 つのモデルに対して自然言語のプロンプトを切り替えるだけ**で、分類・要約・翻訳といった異なるタスクを実行できます（第 1 章 練習問題 4、第 4 章 練習問題 3）。

In [ ]:
tasks = [
    (
        "1) 感情分類",
        [
            {"role": "system", "content": "あなたは宿泊レビューの感情分析アシスタントです。"},
            {
                "role": "user",
                "content": (
                    "次の宿泊レビューの極性を pos（肯定的）、neu（中立）、neg（否定的）の 3 つから 1 つ選び、"
                    "ラベルと簡潔な理由を答えてください。\n\n"
                    "レビュー: 大通り沿いで車の音は聞こえますが、二重窓のおかげで気にならない。"
                ),
            },
        ],
    ),
    (
        "2) 1文要約",
        [
            {"role": "system", "content": "あなたは日本語の要約アシスタントです。"},
            {
                "role": "user",
                "content": (
                    "次の文章を重要なポイントが落ちないように 1 文（60字以内）で要約してください。\n\n"
                    "文章: 事前学習済み大規模言語モデルは、膨大なテキストデータから言語の規則性や知識を獲得しており、"
                    "追加学習を行わなくても自然言語のプロンプトを与えるだけで翻訳や要約、質問応答など多様なタスクに対応できる。"
                    "一方で、事実と異なる内容を自信ありげに生成するハルシネーションが実用上の課題となっている。"
                ),
            },
        ],
    ),
    (
        "3) 英日翻訳",
        [
            {"role": "system", "content": "あなたは技術文書の翻訳アシスタントです。"},
            {
                "role": "user",
                "content": (
                    "次の英文を自然な日本語に翻訳してください。\n\n"
                    "English: While large language models can follow natural language instructions across many tasks, "
                    "grounding their outputs in retrieved documents is essential to reduce hallucinations."
                ),
            },
        ],
    ),
]

for title, messages in tasks:
    ans = generate_response(messages, max_new_tokens=160, temperature=0.0)
    print(f"=== {title} ===")
    print(ans)
    print()

---

## 2. ゼロショット（0-shot）vs フューショット（Few-shot）の比較

自然言語の指示だけで推論する **ゼロショット（0-shot）** に対し、プロンプト内に少数の入出力例を提示する **フューショット（Few-shot / In-Context Learning）** を行うと、タスクの判定基準（特に逆接表現や事実記述の扱い）や出力フォーマットが安定しやすくなります。

ここでは第 5 章の宿泊レビューフレーズデータ（`phrase-annotation.csv`）から `pos` / `neu` / `neg` 各 8 件（計 24 件）を評価セット、重複しない 3 件（各クラス 1 件）を Few-shot 例として、0-shot と 3-shot の以下を比較します。

- **フォーマット遵守率**：余計な文章を混ぜず `pos` / `neu` / `neg` の指定形式で返せた割合
- **Accuracy / Macro-F1**：正解ラベルとの一致精度

In [ ]:
def load_phrase_dataset(csv_path="phrase-annotation.csv"):
    path = Path(csv_path)
    if not path.exists():
        alt = Path("../misc/phrase-annotation.csv")
        if alt.exists():
            path = alt
    df = pd.read_csv(path).fillna({"コメント": ""})
    # 分かち書きスペースを詰めて自然な日本語文にする
    df["text"] = df["フレーズ"].astype(str).str.replace(" ", "", regex=False)
    return df


df_phrases = load_phrase_dataset()
print("全体件数:", len(df_phrases), "| 極性分布:", df_phrases["極性ラベル"].value_counts().to_dict())

# Few-shot 例（各クラス 1 件、逆接や事実記述の判断基準を含む例を選定）
few_shot_ids = [1, 3, 6]
few_shot_df = df_phrases[df_phrases["フレーズID"].isin(few_shot_ids)].copy()
few_shot_examples = list(zip(few_shot_df["text"], few_shot_df["極性ラベル"]))

# 評価セット（Few-shot 例を除いて各クラス 8 件 = 計 24 件を固定シードで抽出）
remain_df = df_phrases[~df_phrases["フレーズID"].isin(few_shot_ids)]
eval_parts = [
    remain_df[remain_df["極性ラベル"] == lbl].sample(n=8, random_state=42)
    for lbl in ["pos", "neu", "neg"]
]
eval_df = pd.concat(eval_parts, ignore_index=True)

print("Few-shot 例:")
for txt, lbl in few_shot_examples:
    print(f"  [{lbl}] {txt}")
print(f"評価セット件数: {len(eval_df)} ({eval_df['極性ラベル'].value_counts().to_dict()})")

In [ ]:
def build_zero_shot_messages(phrase):
    return [
        {
            "role": "system",
            "content": (
                "あなたは日本語の宿泊レビュー極性分類器です。"
                "与えられたフレーズの極性を pos（肯定的）、neu（中立・事実記述のみ）、neg（否定的）の 3 クラスから選び、"
                "必ずラベル（pos / neu / neg）のみを 1 語で出力してください。"
            ),
        },
        {"role": "user", "content": f"フレーズ: {phrase}\nラベル:"},
    ]


def build_few_shot_messages(phrase, examples):
    demo_lines = [
        f"フレーズ: {ex_text}\nラベル: {ex_label}"
        for ex_text, ex_label in examples
    ]
    demo_block = "\n\n".join(demo_lines)
    return [
        {
            "role": "system",
            "content": (
                "あなたは日本語の宿泊レビュー極性分類器です。"
                "以下の例と同じ形式で、与えられたフレーズの極性を pos / neu / neg の 1 語のみで出力してください。"
            ),
        },
        {
            "role": "user",
            "content": f"{demo_block}\n\nフレーズ: {phrase}\nラベル:",
        },
    ]


def parse_sentiment_label(raw_output):
    """LLM 出力から pos/neu/neg を抽出し、(予測ラベル, 厳密フォーマット遵守フラグ) を返す。"""
    cleaned = raw_output.strip().lower()
    exact = cleaned in {"pos", "neu", "neg"}
    if exact:
        return cleaned, True
    match = re.search(r"\b(pos|neu|neg)\b", cleaned)
    if match:
        return match.group(1), False
    if "肯定" in raw_output or "ポジティブ" in raw_output:
        return "pos", False
    if "否定" in raw_output or "ネガティブ" in raw_output:
        return "neg", False
    if "中立" in raw_output:
        return "neu", False
    return "invalid", False


def evaluate_prompt_strategy(df_eval, message_builder):
    preds, raw_outputs, format_ok = [], [], []
    for text in df_eval["text"]:
        msgs = message_builder(text)
        out = generate_response(msgs, max_new_tokens=16, temperature=0.0)
        lbl, ok = parse_sentiment_label(out)
        preds.append(lbl)
        raw_outputs.append(out)
        format_ok.append(ok)

    y_true = df_eval["極性ラベル"].tolist()
    acc = accuracy_score(y_true, preds)
    macro_f1 = f1_score(y_true, preds, labels=["pos", "neu", "neg"], average="macro", zero_division=0)
    compliance = float(np.mean(format_ok))
    return {
        "preds": preds,
        "raw_outputs": raw_outputs,
        "format_compliance": compliance,
        "accuracy": acc,
        "macro_f1": macro_f1,
    }


res_zero = evaluate_prompt_strategy(eval_df, build_zero_shot_messages)
res_few = evaluate_prompt_strategy(eval_df, lambda x: build_few_shot_messages(x, few_shot_examples))

summary_df = pd.DataFrame([
    {
        "プロンプト手法": "Zero-shot (0-shot)",
        "フォーマット遵守率": f"{res_zero['format_compliance']:.1%}",
        "Accuracy": round(res_zero["accuracy"], 4),
        "Macro-F1": round(res_zero["macro_f1"], 4),
    },
    {
        "プロンプト手法": "Few-shot (3-shot)",
        "フォーマット遵守率": f"{res_few['format_compliance']:.1%}",
        "Accuracy": round(res_few["accuracy"], 4),
        "Macro-F1": round(res_few["macro_f1"], 4),
    },
])
display(summary_df)

# 0-shot と 3-shot で予測が分かれた具体例を表示
diff_df = eval_df[["フレーズID", "text", "極性ラベル"]].copy()
diff_df["0-shot予測"] = res_zero["preds"]
diff_df["3-shot予測"] = res_few["preds"]
changed = diff_df[diff_df["0-shot予測"] != diff_df["3-shot予測"]]
print(f"予測が変化した件数: {len(changed)} / {len(eval_df)}")
if len(changed) > 0:
    display(changed.head(8))

---

## 3. ハルシネーションの発生と根拠文書付与（RAG 風プロンプト）による抑制

LLM は事前学習データに含まれない情報（社内規定・架空の施設情報・最新情報など）を尋ねられたとき、**「分からない」と言えずにもっともらしい嘘（ハルシネーション）を生成してしまう**ことがあります（第 1 章 練習問題 5）。

これを抑制する実務上の定番手法が **RAG（Retrieval-Augmented Generation：検索拡張生成）** です。
ここでは架空の宿泊施設「みらいホテル」の館内規定ドキュメントを用意し、以下の 3 つの質問について **「プロンプトのみ（根拠文書なし）」** と **「根拠文書付与（RAG 風プロンプト）」** の回答を比較します。

- **Q1・Q2**：資料に明確な答えが記載されている質問
- **Q3**：資料に記載が**一切ない**質問（屋上プールの営業時間）→ 推測せず **「資料に記載がありません」** と拒否できるかを検証！

In [ ]:
# 架空の宿泊施設「みらいホテル」の館内規定ドキュメント（事前学習データに存在しない独自情報）
knowledge_docs = [
    {
        "id": "doc1",
        "title": "チェックイン・チェックアウト規定",
        "content": "みらいホテルのチェックイン時刻は 15:30 から 22:00 まで、チェックアウト時刻は翌朝 10:30 です。10:30 以降の延長は 1 時間あたり 1,500 円の追加料金が発生します。",
    },
    {
        "id": "doc2",
        "title": "大浴場・サウナの利用時間",
        "content": "みらいホテル 2 階の大浴場は 6:00〜10:00 および 12:30〜24:00 に利用できます。毎日 10:00〜12:30 は清掃・メンテナンス時間帯のため入浴できません。",
    },
    {
        "id": "doc3",
        "title": "ペット同伴宿泊ルール",
        "content": "みらいホテルでは、別館の『ペット専用棟』に限り小型犬（体重 8kg 以下）の同伴宿泊が可能です。ペット宿泊料金は 1 匹あたり 1 泊 3,300 円（税込）で、本館客室への入室はできません。",
    },
    {
        "id": "doc4",
        "title": "キャンセル料規定",
        "content": "宿泊日の 3 日前まではキャンセル料無料です。2 日前は宿泊料金の 30%、前日は 50%、当日および無断不泊は 100% のキャンセル料を申し受けます。",
    },
]


def retrieve_context(query, docs, top_k=2, min_score=0.05):
    """文字 n-gram の TF-IDF コサイン類似度で関連ドキュメントを検索する簡易リトリーバ。"""
    corpus = [f"{d['title']} {d['content']}" for d in docs]
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
    doc_mat = vectorizer.fit_transform(corpus)
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, doc_mat)[0]
    ranked = sorted(zip(docs, scores), key=lambda x: -x[1])
    selected = [(d, float(s)) for d, s in ranked[:top_k] if s >= min_score]
    return selected


def build_direct_qa_messages(question):
    return [
        {"role": "system", "content": "あなたは親切なホテル案内アシスタントです。質問に簡潔に回答してください。"},
        {"role": "user", "content": question},
    ]


def build_rag_qa_messages(question, retrieved_docs):
    if retrieved_docs:
        context_block = "\n".join(
            f"[{d['title']}] {d['content']}" for d, _ in retrieved_docs
        )
    else:
        context_block = "（該当する資料は見つかりませんでした）"

    return [
        {
            "role": "system",
            "content": (
                "あなたは厳密なホテル案内アシスタントです。"
                "必ず『与えられた資料』に記載されている事実のみに基づいて簡潔に回答してください。"
                "与えられた資料に記載がない場合は、推測や一般論で補わず、必ず「資料に記載がありません」と答えてください。"
            ),
        },
        {
            "role": "user",
            "content": f"【与えられた資料】\n{context_block}\n\n【質問】\n{question}",
        },
    ]


qa_cases = [
    ("Q1 (資料あり)", "みらいホテルのチェックイン時刻と、大浴場の清掃時間帯を教えてください。"),
    ("Q2 (資料あり)", "みらいホテルはペット同伴で宿泊できますか？料金も教えてください。"),
    ("Q3 (資料なし・拒否検証)", "みらいホテルの屋上プールの営業時間を教えてください。"),
]

results_qa = []
for q_label, question in qa_cases:
    direct_ans = generate_response(build_direct_qa_messages(question), max_new_tokens=140, temperature=0.0)
    retrieved = retrieve_context(question, knowledge_docs, top_k=2)
    rag_ans = generate_response(build_rag_qa_messages(question, retrieved), max_new_tokens=140, temperature=0.0)

    results_qa.append({
        "質問区分": q_label,
        "質問": question,
        "検索された資料": ", ".join(f"{d['title']}({s:.2f})" for d, s in retrieved) or "なし",
        "根拠文書なし（直接質問）": direct_ans,
        "根拠文書あり（RAG風プロンプト）": rag_ans,
    })

    print(f"=== {q_label}: {question} ===")
    print(f"[根拠なし] {direct_ans}")
    print(f"[検索資料] {', '.join(d['title'] for d, _ in retrieved) or 'なし'}")
    print(f"[RAG 風 ] {rag_ans}")
    print()

pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(results_qa))

---

## 4. まとめ（深層学習・LLM の利点と課題）

本演習を通じて確認した LLM の利点と課題は、書籍 第1章 練習問題 4・5 および 第4章 練習問題 3 の議論と直結しています。

| 観点 | 利点 | 課題・実務上の注意点 |
|---|---|---|
| **タスク汎用性**（1章問4 / 4章問3） | 1 つのモデルに自然言語で指示するだけで分類・要約・翻訳など多様なタスクに対応できる | 出力形式が揺れやすく、システム組み込み時は Few-shot・パーサ・構造化出力の工夫が必要 |
| **事実性・信頼性**（1章問5） | 根拠文書をプロンプトに含める（RAG）ことで、未学習の独自情報にも正確に回答できる | 根拠なしではもっともらしく嘘（ハルシネーション）を生成するため、検索品質と「分からない時は拒否する」設計・評価が不可欠 |
| **コスト・運用** | 少データ・短期間でプロトタイプ（PoC）を立ち上げられる | 推論コスト（GPU メモリ・レイテンシ）が大きく、高頻度・固定タスクでは BERT 等の軽量特化モデル（第 2〜3 章）が有利な場合も多い |